# Baseline — sentence embeddings + logistic regression

Embed each comment with a sentence transformer, then fit one logistic
regression per label. A frozen-embedding baseline — no fine-tuning — for the
fancier models to beat.

In [ ]:
import json
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv('labeled_demo.csv')
# CSV stores "category::label" -> keep just the label name
df['labels'] = df['labels'].apply(lambda s: [x.split('::')[-1] for x in json.loads(s)])

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['labels'])              # rows -> multi-hot label matrix
assert Y.shape == (len(df), len(mlb.classes_))
print(len(df), 'rows,', list(mlb.classes_))

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split

# embed the text once -> frozen feature matrix
embed = SentenceTransformer('all-MiniLM-L6-v2')
X = embed.encode(df['text'].tolist(), normalize_embeddings=True)
assert X.shape[0] == len(df)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# one logistic regression per label, on the embeddings
model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model.fit(X_train, Y_train)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

pred = model.predict(X_test)
print(classification_report(Y_test, pred, target_names=mlb.classes_, zero_division=0))
print('exact-match row accuracy:', round(accuracy_score(Y_test, pred), 3))